# 12-1 导出 Becke 网格数据 (Rust grid-shift Hessian 移植)

为 Rust 端 `src/numint_matmul/hess_rks_becke.rs` 导出 NH3/def2-TZVP 的网格数据。
见计划 `tmp/plan-260820.md` §6。

网格的构造与泛函无关, 因此只导出一份共享文件 `prototype/nh3_grid_becke.npz`
(所有 xc 用例共用):
- `grid_coords, grid_weights`: 网格坐标与 (含 Becke 划分的) 权重, 按生成原子排序
  (`dft.Grids(mol).build(sort_grids=False)`, 对齐 padding 已剔除, `atm_idx >= 0`);
- `quadrature_weights`: 划分前的径向×角向权重;
- `atm_idx`: 每个网格点的生成原子索引 (已按原子分组, 可直接转 ByAtom 边界);
- `radii_table`: Becke 半径调整表 $a_{AB}$ (`grids.radii_adjust`), C-order `[natm, natm]`。

`mo_*`/`ref_de` 不导出: 它们与网格顺序无关, 已存在于 `nh3_r_{xc}.npz`。

## 导出

In [ ]:
import numpy as np
from pyscf import gto, dft

XYZ = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
BASIS = "def2-TZVP"

mol = gto.Mole(atom=XYZ, basis=BASIS, max_memory=8000).build()

grids = dft.Grids(mol).build(sort_grids=False)
# strip alignment padding grids (weights 0, atm_idx -1)
mask = grids.atm_idx >= 0
coords = np.ascontiguousarray(grids.coords[mask])
weights = np.ascontiguousarray(grids.weights[mask])
quadrature_weights = np.ascontiguousarray(grids.quadrature_weights[mask])
atm_idx = np.ascontiguousarray(grids.atm_idx[mask]).astype(np.float64)  # float64: rust read_npz loads f64 only
assert np.all(np.diff(atm_idx) >= 0), "grids not atom-grouped"

natm = mol.natm
becke_scheme = grids.radii_adjust(mol, grids.atomic_radii)
radii_table = np.ascontiguousarray(
    np.array([becke_scheme(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)
)

# sanity: grid set matches the existing reference npz for every xc (order may
# differ; the existing npz may carry alignment-padding zero weights)
for xc in ["svwn", "b3lyp", "tpss0"]:
    old = np.load(f"prototype/nh3_r_{xc}.npz")
    old_nz = np.sort(old["grid_weights"][old["grid_weights"] != 0])
    new_nz = np.sort(weights[weights != 0])
    assert old_nz.shape == new_nz.shape and np.allclose(old_nz, new_nz), f"[{xc}] grid weights mismatch vs existing npz"

out = {
    "grid_coords": coords,
    "grid_weights": weights,
    "quadrature_weights": quadrature_weights,
    "atm_idx": atm_idx,
    "radii_table": radii_table,
}
path = "prototype/nh3_grid_becke.npz"
np.savez(path, **out)
quad_split = np.cumsum(np.bincount(atm_idx.astype(np.int64), minlength=natm))
print(f"wrote {path} (ngrids={coords.shape[0]}, quad_split={quad_split})")